# Proyecto Final: Chatbot de Atención al Cliente para Ripley Chile

## Introducción

El objetivo de este proyecto es desarrollar un chatbot de atención al cliente orientado a Ripley Chile, utilizando modelos de lenguaje y herramientas de LangChain.  

A lo largo del desarrollo se integran distintos conceptos vistos en clases, tales como:
- conexión con modelos de lenguaje,
- uso de mensajes del sistema,
- configuración de parámetros,
- manejo de contexto conversacional,
- streaming de respuestas,
- y memoria de conversación.

El propósito final es construir un asistente virtual capaz de responder consultas frecuentes relacionadas con compras, productos, cambios, devoluciones, despachos y pagos.

## Configuración de Variables de Entorno

Antes de ejecutar el código, asegúrate de tener configuradas las siguientes variables de entorno:

```bash
export GITHUB_BASE_URL="https://models.inference.ai.azure.com"
export GITHUB_TOKEN="tu_token_de_github_aqui"
```

## 1. Instalación de dependencias

Para ejecutar este proyecto se requieren las bibliotecas `openai`, `langchain` y `langchain-openai`.

Ejecutar solo si es necesario
```bash
pip install openai langchain langchain-openai
```

## 2. Importación de bibliotecas

En esta sección se importan las bibliotecas necesarias para configurar el modelo, trabajar con LangChain y gestionar el historial de conversación.

In [11]:
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
import os
import sys
import time

print("OpenAI library version:", __import__('openai').__version__)
print("Python version:", sys.version)
try:
    import langchain
    print(f"LangChain version: {langchain.__version__}")
except ImportError:
    print("LangChain no está instalado")

OpenAI library version: 2.31.0
Python version: 3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]
LangChain version: 1.2.15


## 3. Configuración del entorno y del modelo

A continuación se configuran tanto el cliente OpenAI como el modelo de LangChain, utilizando variables de entorno para evitar exponer credenciales directamente en el código.

In [12]:
# Cliente OpenAI directo
client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

# Modelo principal con LangChain
llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.3,
    max_tokens=150
)

# Modelo con streaming
llm_stream = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    temperature=0.3,
    max_tokens=150
)

print("✓ Cliente OpenAI configurado")
print("✓ Modelo LangChain configurado")
print("✓ Modelo con streaming configurado")

✓ Cliente OpenAI configurado
✓ Modelo LangChain configurado
✓ Modelo con streaming configurado


## 4. Prueba básica del modelo

Esta primera prueba permite verificar que la conexión con el modelo funciona correctamente antes de avanzar hacia la construcción del chatbot.

In [13]:
def llamada_basica():
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": "Preséntate como asistente virtual de Ripley en máximo 2 líneas."
                }
            ],
            temperature=0.1,
            max_tokens=60
        )
        print("=== Respuesta del Modelo ===")
        print(response.choices[0].message.content)
        print("\n=== Información Técnica ===")
        print(f"Modelo usado: {response.model}")
        print(f"Tokens usados: {response.usage.total_tokens}")
        print(f"Tokens de entrada: {response.usage.prompt_tokens}")
        print(f"Tokens de salida: {response.usage.completion_tokens}")
    except Exception as e:
        print(f"Error en la llamada: {e}")
llamada_basica()

=== Respuesta del Modelo ===
¡Hola! Soy el asistente virtual de Ripley, aquí para ayudarte con tus compras, consultas y todo lo que necesites para mejorar tu experiencia. ¡Cuenta conmigo!

=== Información Técnica ===
Modelo usado: gpt-4o-2024-11-20
Tokens usados: 59
Tokens de entrada: 23
Tokens de salida: 36


## 5. Definición del rol del chatbot

El uso del mensaje de sistema permite establecer el comportamiento general del asistente. En este proyecto, el chatbot se orienta a la atención al cliente de Ripley Chile.

In [14]:
def usar_mensaje_sistema():
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Eres un asistente virtual de atención al cliente de Ripley Chile. "
                        "Ayudas con dudas sobre compras, productos, despachos, devoluciones, cambios y pagos. "
                        "Respondes de forma clara, amable y breve. "
                        "Si no sabes algo con certeza, indícalo sin inventar información."
                    )
                },
                {
                    "role": "user",
                    "content": "¿Cómo puedo devolver un producto comprado online?"
                }
            ],
            temperature=0.3,
            max_tokens=120
        )
        print("=== Respuesta con Mensaje de Sistema ===")
        print(response.choices[0].message.content)
        print(f"\nTokens usados: {response.usage.total_tokens}")
    except Exception as e:
        print(f"Error: {e}")
usar_mensaje_sistema()

=== Respuesta con Mensaje de Sistema ===
Para devolver un producto comprado online en Ripley Chile, sigue estos pasos:

1. **Revisa las condiciones de devolución**: Asegúrate de que el producto esté dentro del plazo de devolución (generalmente 10 días desde la recepción) y en las condiciones originales, sin uso y con sus etiquetas y embalaje.

2. **Solicita la devolución**: Ingresa a tu cuenta en el sitio web de Ripley y selecciona la opción "Mis Compras". Busca el producto que deseas devolver y sigue las instrucciones para generar la solicitud de devolución.

3. **Elige el

Tokens usados: 200


## 6. Exploración del parámetro temperature

El parámetro `temperature` permite controlar el nivel de variabilidad de las respuestas. En un chatbot de atención al cliente, temperaturas bajas o moderadas suelen ser más adecuadas, ya que ayudan a mantener respuestas consistentes y confiables.

In [15]:
def comparar_temperature():
    prompt = "Un cliente está molesto porque su pedido no ha llegado. Responde como asistente de Ripley para ayudarlo y tranquilizarlo."
    
    temperatures = [0.1, 0.7]
    for temp in temperatures:
        print(f"\n{'='*50}")
        print(f"TEMPERATURE: {temp}")
        print('='*50)
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Eres un asistente virtual de Ripley Chile. "
                            "Respondes de forma clara, amable y útil."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=temp,
                max_tokens=80
            )
            print(response.choices[0].message.content)
            print(f"\nTokens usados: {response.usage.total_tokens}")
        except Exception as e:
            print(f"Error: {e}")
comparar_temperature()


TEMPERATURE: 0.1
¡Hola! Lamento mucho que tu pedido aún no haya llegado. Entendemos lo importante que es para ti recibir tus productos a tiempo y queremos ayudarte a resolver esta situación lo antes posible.

Por favor, ¿podrías compartir conmigo el número de tu pedido o algún dato adicional, como tu nombre completo o el correo electrónico asociado a la compra? Así podremos revisar el estado de tu envío y

Tokens usados: 141

TEMPERATURE: 0.7
¡Hola! Lamento mucho que estés pasando por esta situación y que tu pedido aún no haya llegado. Estoy aquí para ayudarte y resolver este inconveniente lo antes posible.

Por favor, ¿podrías proporcionarme los siguientes datos? 
1. Número de pedido o algún dato relacionado con tu compra.  
2. Nombre completo del titular de la compra.  

Con esta información, podré revisar

Tokens usados: 141


## 7. Integración con LangChain

LangChain permite trabajar con una estructura conversacional más organizada, utilizando distintos tipos de mensajes para representar el contexto de una conversación.

In [16]:
def ejemplo_basico_langchain():
    try:
        response = llm.invoke([
            HumanMessage(content="Preséntate como asistente virtual de Ripley en máximo 2 líneas.")
        ])
        print("=== Respuesta Básica con LangChain ===")
        print(response.content)
        print(f"Tipo de respuesta: {type(response)}")
    except Exception as e:
        print(f"Error: {e}")
ejemplo_basico_langchain()

=== Respuesta Básica con LangChain ===
¡Hola! Soy el asistente virtual de Ripley, aquí para ayudarte con tus compras, consultas y todo lo que necesites. ¿En qué puedo asistirte hoy?
Tipo de respuesta: <class 'langchain_core.messages.ai.AIMessage'>


## 8. Conversación con contexto

Una de las ventajas de LangChain es que permite construir conversaciones más estructuradas, manteniendo el contexto previo entre mensajes.

In [17]:
def ejemplo_conversacion_completa():
    try:
        messages = [
            SystemMessage(content=(
                "Eres un asistente virtual de atención al cliente de Ripley Chile. "
                "Respondes de manera clara, amable y útil. "
                "Ayudas con compras, productos, cambios, devoluciones y despachos."
            )),
            HumanMessage(content="Compré una chaqueta y quiero cambiarla."),
            AIMessage(content="Claro, puedo orientarte con eso. ¿La compra fue realizada en tienda física o por internet?"),
            HumanMessage(content="La compré por internet.")
        ]
        response = llm.invoke(messages)
        print("=== Conversación con Contexto ===")
        print(response.content)
    except Exception as e:
        print(f"Error: {e}")
ejemplo_conversacion_completa()

=== Conversación con Contexto ===
¡Entendido! Para cambiar una chaqueta comprada por internet, primero verifica que cumpla con las condiciones de cambio:

1. **Plazo:** Tienes hasta **30 días** desde la recepción del producto para realizar el cambio.
2. **Estado del producto:** La chaqueta debe estar sin uso, con etiquetas originales y en su empaque original.
3. **Documentación:** Necesitarás la boleta o comprobante de compra.

Puedes realizar el cambio de dos maneras:

### 1. **En tienda física:**
   - Lleva la chaqueta junto con la boleta o comprobante de compra a cualquier tienda Ripley.
   - Dirígete al área de Servicio al Cliente para gestionar el cambio.

### 


## 9. Incorporación de memoria conversacional

Para acercar el proyecto a un chatbot más real, se incorpora memoria conversacional. Esto permite que el asistente recuerde mensajes previos dentro de una misma sesión.

In [18]:
prompt_memoria = ChatPromptTemplate.from_messages([
    ("system", 
     "Eres un asistente virtual de atención al cliente de Ripley Chile. "
     "Ayudas con dudas sobre compras, cambios, devoluciones, pagos, productos y despachos. "
     "Respondes de forma clara, breve y útil."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt_memoria | llm

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

In [19]:
def ejemplo_memoria():
    print("=== CHATBOT CON MEMORIA ===")
    
    session_id = "cliente_ripley"
    try:
        print("\n1. Primera interacción:")
        response1 = conversation.invoke(
            {"input": "Compré un notebook en Ripley y aún no me llega."},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response1.content}\n")

        print("2. Segunda interacción:")
        response2 = conversation.invoke(
            {"input": "¿Qué problema te comenté recién?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response2.content}\n")

        print("3. Tercera interacción:")
        response3 = conversation.invoke(
            {"input": "¿Qué producto mencioné?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response3.content}\n")

        print("=== HISTORIAL DE LA MEMORIA ===")
        history = store[session_id].messages
        for i, msg in enumerate(history, 1):
            print(f"{i}. {msg.type}: {msg.content}")
    except Exception as e:
        print(f"Error: {e}")
ejemplo_memoria()

=== CHATBOT CON MEMORIA ===

1. Primera interacción:
Respuesta: Lamento que aún no hayas recibido tu notebook. Para ayudarte, por favor verifica lo siguiente:

1. **Estado del despacho**: Ingresa a [Mi Cuenta](https://www.ripley.cl/mi-cuenta) en el sitio web de Ripley y revisa el estado de tu pedido en la sección "Mis compras".
   
2. **Plazo de entrega**: Confirma si el plazo estimado de despacho ya venció. Este plazo está indicado en el correo de confirmación de compra.

3. **Contacto con soporte**: Si el plazo ya venció o necesitas más información, puedes comunicarte con nuestro servicio al cliente al número **600 200 0047** o escribirnos a través del chat en el

2. Segunda interacción:
Respuesta: Me comentaste que compraste un notebook en Ripley y aún no te llega. ¿Te ayudo a revisar el estado de tu pedido o necesitas más información sobre el despacho?

3. Tercera interacción:
Respuesta: Mencionaste que compraste un **notebook**. ¿Puedo ayudarte a rastrear el estado de tu pedido o 

## 10. Streaming de respuestas

El streaming permite mostrar la respuesta del chatbot en tiempo real, mejorando la percepción de rapidez e interacción del usuario.

In [20]:
def streaming_basico():
    prompt = "Explícame brevemente cómo un chatbot de Ripley puede ayudar a un cliente."
    
    print("=== STREAMING EN TIEMPO REAL ===")
    print("Generando respuesta...")
    print("-" * 50)
    try:
        for chunk in llm_stream.stream([HumanMessage(content=prompt)]):
            print(chunk.content, end="", flush=True)
            time.sleep(0.01)
            
        print("\n" + "-" * 50)
        print("✓ Streaming completado")
    except Exception as e:
        print(f"✗ Error en streaming: {e}")
streaming_basico()

=== STREAMING EN TIEMPO REAL ===
Generando respuesta...
--------------------------------------------------
Un chatbot de Ripley puede ayudar a un cliente de manera rápida y eficiente en diversas áreas, como:

1. **Información sobre productos**: Responde preguntas sobre características, precios, disponibilidad y promociones de productos en la tienda.

2. **Seguimiento de pedidos**: Permite a los clientes consultar el estado de sus compras, fechas de entrega y detalles de envío.

3. **Resolución de problemas**: Brinda soporte en caso de inconvenientes con pedidos, devoluciones, cambios o reembolsos.

4. **Asesoramiento personalizado**: Ayuda a los clientes a encontrar productos según sus necesidades y preferencias.

5. **Ubicación de tiendas**: Proporciona información sobre direcciones, horarios y servicios disponibles en
--------------------------------------------------
✓ Streaming completado


## 11. Implementación del chatbot final

Finalmente, se implementa una versión funcional del chatbot en consola, integrando:
- contexto conversacional,
- memoria por sesión,
- y respuestas en streaming.

Esta sección representa la versión más cercana al objetivo final del proyecto.

In [ ]:
def chatbot_ripley():
    print("=== CHATBOT RIPLEY CHILE ===")
    print("Escribe 'salir' para terminar la conversación.\n")
    
    session_id = "chat_ripley"
    while True:
        user_input = input("🧑 Tú: ")
        
        if user_input.lower() in ["salir", "exit", "quit"]:
            print("\n👋 ¡Gracias por usar el chatbot de Ripley!")
            break
        if not user_input.strip():
            continue
        print("\n🤖 RipleyBot: ", end="", flush=True)
        try:
            # Guardar mensaje en memoria
            history = get_session_history(session_id)
            # Construir mensajes manualmente para streaming
            messages = [
                SystemMessage(content=(
                    "Eres un asistente virtual de atención al cliente de Ripley Chile. "
                    "Ayudas con dudas sobre compras, productos, despachos, devoluciones, cambios y pagos. "
                    "Respondes de forma clara, amable y breve. "
                    "Si no sabes algo con certeza, indícalo sin inventar información."
                ))
            ] + history.messages + [HumanMessage(content=user_input)]
            
            full_response = ""
            for chunk in llm_stream.stream(messages):
                content = chunk.content
                print(content, end="", flush=True)
                full_response += content
                time.sleep(0.01)
            print()
            # Guardar conversación en memoria
            history.add_user_message(user_input)
            history.add_ai_message(full_response)
        except KeyboardInterrupt:
            print("\n\n⏸️ Conversación interrumpida")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")
# Descomentar para probar
# chatbot_ripley()

=== CHATBOT RIPLEY CHILE ===
Escribe 'salir' para terminar la conversación.


🤖 RipleyBot: ¡Hola! 😊 ¿En qué puedo ayudarte hoy?

🤖 RipleyBot: Para devolver un producto en Ripley Chile, sigue estos pasos:

1. **Revisa las condiciones de devolución:** Asegúrate de que el producto esté en buen estado, sin uso, con etiquetas y en su empaque original. Además, verifica que esté dentro del plazo de devolución (generalmente 10 días desde la recepción).

2. **Solicita la devolución:** 
   - Si compraste en **ripley.cl**, inicia el proceso en la sección "Mis Compras" de tu cuenta. Busca el pedido, selecciona el producto y sigue las instrucciones para solicitar la devolución.
   - Si compraste en una **tienda física**, dirígete a la tienda con el producto, boleta o comprobante de compra.



👋 ¡Gracias por usar el chatbot de Ripley!


## Conclusión

En este proyecto se desarrolló un prototipo de chatbot de atención al cliente para Ripley Chile, integrando conceptos fundamentales vistos durante el curso.  

A lo largo del trabajo se configuró la conexión con modelos de lenguaje, se exploró el uso de mensajes del sistema, parámetros como `temperature`, estructuras conversacionales con LangChain, memoria de conversación y streaming de respuestas.

Como resultado, se obtuvo una base funcional para un asistente virtual capaz de responder consultas frecuentes dentro del contexto de atención al cliente.